# Citation Hallucination — Local Model Inference Pipeline

**Dataset:** `true_false_baseline_complete.csv` (Single prompt per row)

**Models (1-8B Scale):**
| # | Model | HF ID |
|---|-------|-------|
| 1 | Llama 3.2-3B-Instruct | meta-llama/Llama-3.2-3B-Instruct |
| 2 | Gemma 3-4B-IT | google/gemma-3-4b-it |
| 3 | Phi-4-mini-Instruct | microsoft/Phi-4-mini-instruct |
| 4 | DeepSeek-LLM-7B-Chat | deepseek-ai/deepseek-llm-7b-chat |
| 5 | Mistral-7B-Instruct-v0.3 | mistralai/Mistral-7B-Instruct-v0.3 |
| 6 | Llama 3.1-8B-Instruct | meta-llama/Llama-3.1-8B-Instruct |


---\n## 1. Setup & Imports

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install",
    "protobuf==3.20.3", "vllm", "transformers", "accelerate", "bitsandbytes",
    "pandas", "matplotlib", "seaborn", "tqdm"])

import gc, json, os, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datetime import datetime
from tqdm.auto import tqdm
from collections import defaultdict
from pathlib import Path

RUN_ID = "1"
OUTPUT_DIR = "outputs/inference_results"
METRICS_DIR = "outputs/metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

print(f"RUN_ID: {RUN_ID}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


RUN_ID: 1
GPU: NVIDIA RTX 6000 Ada Generation


---\n## 2. Model Registry

In [2]:
MODEL_REGISTRY = {
    "Llama-3.1-8B-Instruct": {
        "hf_id": "meta-llama/Llama-3.1-8B-Instruct",
        "quantize": False,
        "gpu_memory_utilization": 0.95,
    },
    "Mistral-7B-Instruct-v0.3": {
        "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "quantize": False,
        "gpu_memory_utilization": 0.95,
    },
    "Phi-4-mini-Instruct": {
        "hf_id": "microsoft/Phi-4-mini-instruct",
        "quantize": False,
        "gpu_memory_utilization": 0.95,
    },
    "DeepSeek-LLM-7B-Chat": {
        "hf_id": "deepseek-ai/deepseek-llm-7b-chat",
        "quantize": False,
        "gpu_memory_utilization": 0.95,
    }
}

MODELS_TO_RUN = list(MODEL_REGISTRY.keys())
print(f"Models loaded in registry: {len(MODEL_REGISTRY)}")


Models loaded in registry: 4


---
## 3. Load Dataset

Set `SAMPLE_SIZE = None` for the full 110K rows, or a number for a stratified sample.

In [3]:
CSV_PATH = "true_false_baseline_sorted.csv"
SAMPLE_SIZE = None  # None = full dataset, or e.g. 5000 for a quick test
SEED = 42

print(f"Loading dataset from {CSV_PATH}...")
df_data = pd.read_csv(CSV_PATH, low_memory=False)
if 'source_label' in df_data.columns:
    df_data['source_label'] = df_data['source_label'].astype(str).str.strip().str.lower()
print(f"  Loaded {len(df_data):,} rows, {len(df_data.columns)} columns")

if SAMPLE_SIZE and SAMPLE_SIZE < len(df_data):
    print(f"  Sampling {SAMPLE_SIZE} rows...")
    df_data = df_data.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    print(f"  Sampled {len(df_data)} rows")

# This is a single-prompt dataset
PROMPT_COLUMN = "prompt"
if PROMPT_COLUMN not in df_data.columns:
    raise ValueError(f"Could not find '{PROMPT_COLUMN}' column in dataset! Columns: {df_data.columns.tolist()}")

print(f"\n  Total inferences needed: ~{len(df_data) * len(MODELS_TO_RUN):,} "
      f"({len(df_data)} rows × {len(MODELS_TO_RUN)} models)")

# Convert to a list of dicts for faster iteration later
rows = df_data.to_dict(orient="records")
total_rows = len(rows)


Loading dataset from true_false_baseline_sorted.csv...
  Loaded 330,846 rows, 29 columns

  Total inferences needed: ~1,323,384 (330846 rows × 4 models)


---
## 4. Inference Helpers & Setup

Using vLLM for high-speed batched generation.

In [4]:
def format_prompts(tokenizer, texts, max_len=4096, max_new_tokens=512):
    """Format raw texts into chat templates and ensure they don't exceed max context length."""
    formatted = []

    # Leave room for generation (safety margin)
    allowed_input_len = max_len - max_new_tokens - 20

    for t in texts:
        # 1. Truncate raw text if it's monstrously long to prevent tokenization overflow
        # Very rough heuristic: 1 token ~= 4 chars
        if len(t) > allowed_input_len * 4:
            t = t[:allowed_input_len * 4]

        # 2. Tokenize, strict truncate, decode
        # Using built-in truncation to avoid "Token indices sequence length is longer than..." warnings
        tokens = tokenizer.encode(t, add_special_tokens=False, truncation=True, max_length=allowed_input_len)
        t = tokenizer.decode(tokens)

        messages = [{"role": "user", "content": t}]
        try:
            # We check if tokenizer has apply_chat_template natively
            if hasattr(tokenizer, "apply_chat_template"):
                formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            else:
                formatted.append(t)
        except:
            formatted.append(t)
    return formatted

def unload_vllm(llm):
    """Properly unload vLLM to free GPU memory for the next model."""
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment
        destroy_model_parallel()
        destroy_distributed_environment()
    except:
        pass
    print("  → Model unloaded, GPU memory freed")
    time.sleep(2)

print("Helper functions defined ✓")

Helper functions defined ✓


---
## 5. Run Inference (All Models)

For each model: build prompts → load vLLM → generate batched → save → unload.

In [ ]:
all_results = {}

BATCH_SIZE = 3308

for model_name in MODELS_TO_RUN:
    model_info = MODEL_REGISTRY[model_name]
    hf_id = model_info["hf_id"]
    output_path = os.path.join(OUTPUT_DIR, f"{model_name}_{RUN_ID}_final.csv")

    # 1. Load Checkpoint Data by strict row count
    completed_count = 0
    if os.path.exists(output_path):
        try:
            import pandas as pd
            # Use pandas to safely read CSV rows (handles quoted newlines and headers properly)
            completed_count = len(pd.read_csv(output_path, usecols=[0]))
            print(f"  → Resuming from checkpoint: {completed_count} rows already evaluated.")
        except Exception as e:
            print(f"  → Warning: Checkpoint corrupted or not readable: {e}")

    # 2. Build Uncompleted Requests
    valid_count = 0
    requests = []
    for idx, row in df_data.iterrows():
        # PASS 1 OPTIMIZATION: Skip the massive 19k outliers for now so we can use a fast batch size
        if len(str(row.get("prompt", ""))) > 15000:
            continue

        if pd.isna(row.get("prompt")):
            continue

        # ONLY process valid rows we haven't generated yet
        if valid_count < completed_count:
            valid_count += 1
            continue

        requests.append({
            "instance_id": str(row["instance_id"]),
            "prompt_text": str(row["prompt"]),
            "meta": row.to_dict()
        })

    print(f"  Prepared {len(requests)} remaining task requests out of {len(df_data)} total dataset rows.")
    if not requests:
        print(f"  Skipping model generation: No remaining requests.")
        continue

    # 3. Load vLLM
    print(f"  Loading LLM...")
    dtype_str = model_info.get("dtype", "bfloat16")

    # PASS 1 CONFIG: Fast batching, max context 4096
    max_len = 19800

    from vllm import LLM, SamplingParams
    llm_kwargs = {
        "model": hf_id,
        "dtype": dtype_str,
        "max_model_len": max_len,
        "gpu_memory_utilization": model_info.get("gpu_memory_utilization", 0.90),
        "enforce_eager": True,
    }
    llm = LLM(**llm_kwargs)
    tokenizer = llm.get_tokenizer()
    sampling_params = SamplingParams(temperature=0.0, max_tokens=512)

    # 4. Generate in Chunks
    total_requests = len(requests)
    print(f"  Starting chunked pipeline ({total_requests} prompts, {BATCH_SIZE}/chunk)...")
    start_time = time.time()

    results = []
    errors = 0

    # Replaced open with Pandas CSV to_csv mode=a below
    for i in range(0, total_requests, BATCH_SIZE):
        chunk_reqs = requests[i : i + BATCH_SIZE]
        print(f"\nProcessing Chunk {i//BATCH_SIZE + 1} (Rows {i} to {min(i + BATCH_SIZE, total_requests)})...")

        raw_prompts = [str(r["prompt_text"]) for r in chunk_reqs]
        print("    Submitting to GPU...")

        chunk_prompts = format_prompts(tokenizer, raw_prompts, max_len=max_len, max_new_tokens=512)
        vllm_outputs = llm.generate(chunk_prompts, sampling_params)
        chunk_outputs_parsed = [
            {"text": o.outputs[0].text.strip() if o.outputs else "ERROR: No output generated"}
            for o in vllm_outputs
        ]

        for req, output in zip(chunk_reqs, chunk_outputs_parsed):
            resp = output["text"]
            if resp.startswith("ERROR:"):
                errors += 1

            # Keep all metadata from master dataset natively
            res = {k: v for k, v in req["meta"].items() if k not in ["prompt", "prompt_text"]}
            res["instance_id"] = req["instance_id"]
            res["model_name"] = model_name
            res["response"] = resp
            res["response_len"] = len(resp)
            results.append(res)

        # Save chunk to CSV incrementally
        import pandas as pd
        import os
        chunk_df = pd.DataFrame(results[-len(chunk_reqs):])
        if not os.path.exists(output_path):
            chunk_df.to_csv(output_path, index=False)
        else:
            chunk_df.to_csv(output_path, mode="a", header=False, index=False)
        print(f"  → Finished chunk {i//BATCH_SIZE + 1}. Progress: {min(i + BATCH_SIZE, total_requests)}/{total_requests}")

    elapsed = time.time() - start_time
    print(f"  → Done! {len(results)} new results, {errors} errors, {elapsed/60:.1f} min")

    # 5. Unload
    unload_vllm(llm)
    all_results[model_name] = results


  → Resuming from checkpoint: 115773 rows already evaluated.
  Prepared 215031 remaining task requests out of 330846 total dataset rows.
  Loading LLM...
INFO 04-09 19:56:27 [utils.py:261] non-default args: {'dtype': 'bfloat16', 'max_model_len': 19800, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
INFO 04-09 19:56:29 [model.py:541] Resolved architecture: LlamaForCausalLM
INFO 04-09 19:56:29 [model.py:1561] Using max model len 19800
INFO 04-09 19:56:29 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-09 19:56:29 [vllm.py:624] Asynchronous scheduling is enabled.
WARNING 04-09 19:56:29 [vllm.py:662] Enforce eager set, overriding optimization level to -O0
INFO 04-09 19:56:29 [vllm.py:762] Cudagraph is disabled under eager mode
WARNING 04-09 19:56:31 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spaw

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:00,  6.54it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:00<00:00,  2.13it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:01<00:00,  1.50it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.33it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.52it/s]
(EngineCore_DP0 pid=1805685) 


(EngineCore_DP0 pid=1805685) INFO 04-09 19:56:46 [default_loader.py:291] Loading weights took 2.90 seconds
(EngineCore_DP0 pid=1805685) INFO 04-09 19:56:46 [gpu_model_runner.py:4118] Model loading took 14.99 GiB memory and 5.236562 seconds
(EngineCore_DP0 pid=1805685) INFO 04-09 19:56:48 [gpu_worker.py:356] Available KV cache memory: 28.86 GiB
(EngineCore_DP0 pid=1805685) INFO 04-09 19:56:48 [kv_cache_utils.py:1307] GPU KV cache size: 236,448 tokens
(EngineCore_DP0 pid=1805685) INFO 04-09 19:56:48 [kv_cache_utils.py:1312] Maximum concurrency for 19,800 tokens per request: 11.94x
(EngineCore_DP0 pid=1805685) INFO 04-09 19:56:48 [core.py:272] init engine (profile, create kv cache, warmup model) took 2.18 seconds
(EngineCore_DP0 pid=1805685) INFO 04-09 19:56:51 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=1805685) WARNING 04-09 19:56:51 [vllm.py:669] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor com

Adding requests:   0%|          | 0/3308 [00:00<?, ?it/s]

Processed prompts:   0%|                   | 0/3308 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.…

...

Adding requests:   0%|          | 0/3308 [00:00<?, ?it/s]

Processed prompts:   0%|                   | 0/3308 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.…

  → Finished chunk 33. Progress: 109164/215031

Processing Chunk 34 (Rows 109164 to 112472)...
    Submitting to GPU...


Adding requests:   0%|          | 0/3308 [00:00<?, ?it/s]

Processed prompts:   0%|                   | 0/3308 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.…

In [ ]:
# Load all final JSONL files from OUTPUT_DIR
all_records = []
for fpath in sorted(Path(OUTPUT_DIR).glob(f"*_{RUN_ID}_final.csv")):
    print(f"  Loading: {fpath.name}")
    df_temp = pd.read_csv(fpath, low_memory=False)
    all_records.extend(df_temp.to_dict("records"))

# If no files found for this RUN_ID, try loading all final files
if not all_records:
    print(f"  No files for RUN_ID={RUN_ID}, loading all *_final.csv...")
    for fpath in sorted(Path(OUTPUT_DIR).glob("*_final.csv")):
        print(f"  Loading: {fpath.name}")
        import pandas as pd
        df_temp = pd.read_csv(fpath, low_memory=False)
        all_records.extend(df_temp.to_dict("records"))

df_results = pd.DataFrame(all_records)
print(f"\n
Total records: {len(df_results)}")
print(f"Models:        {sorted(df_results['model_name'].unique())}")
print(f"Conditions:    {sorted(df_results['condition'].unique())}")

---
## 8. Response Statistics

Basic stats on the raw responses. Judging/hallucination analysis is a **separate step**.

In [ ]:
# Add response_len if not present
if "response_len" not in df_results.columns:
    df_results["response_len"] = df_results["response"].str.len()

# Flag errors
df_results["is_error"] = df_results["response"].str.startswith("ERROR:", na=False)

models = sorted(df_results["model_name"].unique())

print("=" * 70)
print("RESPONSE STATISTICS BY MODEL × CONDITION")
print("=" * 70)
for model in models:
    df_m = df_results[df_results["model_name"] == model]
    print(f"\n
  {model}:")
    for cond in ["no_citation", "with_citation"]:
        df_c = df_m[df_m["condition"] == cond]
        n = len(df_c)
        n_err = df_c["is_error"].sum()
        avg_len = df_c.loc[~df_c["is_error"], "response_len"].mean()
        print(f"    {cond:15s}: n={n:>6}  errors={n_err:>4}  avg_response_len={avg_len:,.0f} chars")

print(f"\n
  Total records: {len(df_results)}")
print(f"  Total errors:  {df_results['is_error'].sum()}")

---
## 9. Save All Results

Results saved as JSONL (per model) and a combined CSV.
These will be input to the judging pipeline.

In [ ]:
# Save combined CSV
combined_csv = os.path.join(OUTPUT_DIR, f"all_responses_{RUN_ID}.csv")
df_results.to_csv(combined_csv, index=False)
print(f"Saved combined CSV: {combined_csv}")
print(f"  {len(df_results)} records, {len(df_results.columns)} columns")

# List all output files
print(f"\n
All output files:")
for f in sorted(Path(OUTPUT_DIR).glob(f"*{RUN_ID}*")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} MB)")

---
## 10. Sample Responses (sanity check)

Show a few responses per model to verify outputs look reasonable.

In [ ]:
for model in models:
    df_m = df_results[df_results["model_name"] == model]
    print(f"\n
{'='*70}")
    print(f"MODEL: {model}")
    print(f"{'='*70}")

    # Show 2 samples per condition
    for cond in ["no_citation", "with_citation"]:
        df_c = df_m[df_m["condition"] == cond].head(2)
        for _, row in df_c.iterrows():
            print(f"\n
  [{cond}] {row['instance_id']} (truth={row.get('claim_truth','?')})")
            print(f"    Prompt:   {row.get('prompt_preview','')[:120]}...")
            resp = str(row.get("response", ""))
            print(f"    Response: {resp[:300]}{'...' if len(resp) > 300 else ''}")

---\n## 11. Final Summary

In [ ]:
print("=" * 70)
print("INFERENCE PIPELINE — COMPLETE")
print("=" * 70)
print(f"Run ID:          {RUN_ID}")
print(f"Dataset:         {CSV_PATH}")
print(f"Rows:            {len(df_data)}")
print(f"Models:          {MODELS_TO_RUN}")
print(f"Total responses: {len(df_results)}")
print(f"Total errors:    {df_results['is_error'].sum()}")
print()

# Per-model summary
summary_rows = []
for model in models:
    df_m = df_results[df_results["model_name"] == model]
    n_total = len(df_m)
    n_err = df_m["is_error"].sum()
    avg_len = df_m.loc[~df_m["is_error"], "response_len"].mean()
    summary_rows.append({
        "Model": model,
        "Responses": n_total,
        "Errors": int(n_err),
        "Avg Response Len": f"{avg_len:,.0f}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

print(f"\n
Output directory: {OUTPUT_DIR}/")
print(f"Combined CSV:     {OUTPUT_DIR}/all_responses_{RUN_ID}.csv")
print(f"\n
Next step: run the judging pipeline on these responses.")